In [49]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [50]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
# os.environ["GROQ_API_KEY"]

In [51]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=os.getenv("GROQ_API_KEY"))
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A1C99BE980>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A1C99BFE50>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [52]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="You are a helpful assistant that translates English to French."),
    HumanMessage(content="'Hello, how are you?'")
]

In [53]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()

In [54]:
raw_response = model.invoke(messages)
raw_response

AIMessage(content='"Bonjour, comment allez-vous?"', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 53, 'total_tokens': 62, 'completion_time': 0.00634365, 'completion_tokens_details': None, 'prompt_time': 0.003365408, 'prompt_tokens_details': None, 'queue_time': 0.050733682, 'total_time': 0.009709058}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c0a69-c193-71a2-98e3-5baca7ff7d48-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 53, 'output_tokens': 9, 'total_tokens': 62})

In [55]:
parsered_response = parser.invoke(raw_response)
parsered_response

'"Bonjour, comment allez-vous?"'

In [56]:
## Using Langchain Execution Chain
chain = model | parser
chain.invoke(messages)

'"Bonjour, comment allez-vous ?"'

In [57]:
from langchain_core.prompts import ChatPromptTemplate

generic_prompt = "You are a helpful assistant that translates English to {language}."

prompt = ChatPromptTemplate.from_messages([
    ("system", generic_prompt),
    ("user", "{input}")
])

In [58]:
prompt_results =prompt.invoke({"language":"french","input": "Hello, how are you?"})
prompt_results

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant that translates English to french.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={})])

In [59]:
prompt_results.to_messages()

[SystemMessage(content='You are a helpful assistant that translates English to french.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={})]

In [60]:
chain = prompt | model | parser
chain.invoke({"language":"french","input": "Hello, how are you?"})

"Bonjour, je vais bien, merci. Comment allez-vous ? \n\n(Translation: Hello, I'm fine, thank you. How are you?)"